In [9]:
import json
import nltk
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Pastikan NLTK memiliki resource yang dibutuhkan
nltk.download('punkt')
nltk.download('wordnet')

# Load file intents_fix.json
with open('intents_fix.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Inisialisasi tokenizer dan lemmatizer
lemmatizer = nltk.WordNetLemmatizer()
tokenizer = Tokenizer()

# Variabel untuk menyimpan data
sentences = []
labels = []
label_map = {}
responses = {}

# Proses parsing intents.json
for intent in data['intents']:
    if intent['tag'] not in label_map:
        label_map[intent['tag']] = len(label_map)
        responses[label_map[intent['tag']]] = intent['responses']
    for pattern in intent['patterns']:
        words = nltk.word_tokenize(pattern)
        words = [lemmatizer.lemmatize(word.lower()) for word in words]
        sentences.append(' '.join(words))
        labels.append(label_map[intent['tag']])

# Tokenisasi teks
tokenizer.fit_on_texts(sentences)
X = tokenizer.texts_to_sequences(sentences)
X = pad_sequences(X, padding='post')

# Konversi label ke array numpy
y = np.array(labels)

# Split data menjadi training (80%) dan testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Parameter model
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 16
max_length = X.shape[1]

# Definisi model LSTM
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    LSTM(64, return_sequences=True),
    LSTM(64),
    Dense(64, activation='relu'),
    Dense(len(label_map), activation='softmax')
])

# Kompilasi model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Pelatihan model
model.fit(X_train, y_train, epochs=50, batch_size=8, validation_data=(X_test, y_test))

# Simpan model, tokenizer, dan respons
model.save("chatbot_lstm_model.h5")
with open('tokenizer.json', 'w', encoding='utf-8') as tokenizer_file:
    json.dump(tokenizer.to_json(), tokenizer_file)

with open('label_map.json', 'w', encoding='utf-8') as label_map_file:
    json.dump(label_map, label_map_file)

with open('responses.json', 'w', encoding='utf-8') as responses_file:
    json.dump(responses, responses_file)

print("Model, tokenizer, dan respons berhasil disimpan!")


Epoch 1/50


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\maeru\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\maeru\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.0297 - loss: 3.1356 - val_accuracy: 0.0323 - val_loss: 3.1369
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0826 - loss: 3.1250 - val_accuracy: 0.0000e+00 - val_loss: 3.1434
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0226 - loss: 3.0998 - val_accuracy: 0.0968 - val_loss: 3.2256
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0417 - loss: 3.0118 - val_accuracy: 0.0323 - val_loss: 3.1037
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1507 - loss: 2.9617 - val_accuracy: 0.0645 - val_loss: 3.1395
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1229 - loss: 2.7928 - val_accuracy: 0.0323 - val_loss: 2.9272
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0824 - loss: 2.6766 - val_accuracy: 0.1613 - val_loss: 3.0361
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1125 - loss: 2.5672 - val_accuracy: 0.0645 - val_loss: 2.796

Model, tokenizer, dan respons berhasil disimpan!
